# 02 · CRISPR benchmark: Table 1a (K562, HCT116, GM12878) and Table 1c (by distance)

Held-out CRISPR element–gene pairs from the ENCODE CRISPR benchmark: K562 pairs from five studies not used in training (1,895 pairs, 115 positives), HCT116 (373, 34) and GM12878 (68, 16). CisLink scores are computed here from the shipped feature tables; ENCODE-rE2G, ABC and distance come with the pairs. 95% CIs: gene-clustered bootstrap (1,000 replicates).

The hFOB CRISPRi and STING-seq screens of BMD variants are not redistributed here; see `data/README.md`.

In [1]:
import numpy as np
import pandas as pd

from cislink import datasets
from cislink import evaluate as ev

M = ["CisLink-E2G", "CisLink-V2G", "ENCODE-rE2G Extended", "ENCODE-rE2G", "ABC", "Distance"]
P = {c: datasets.crispr_pairs(c) for c in datasets.CELLS}
rows = []
for c, p in P.items():
    r = {"set": c, "pairs": len(p), "positives": int(p.label.sum())}
    for m in M:
        r[m] = np.nan if p[m].isna().all() else ev.auprc(p.label, p[m])
    rows.append(r)
pd.DataFrame(rows).round(3)

,set,pairs,positives,CisLink-E2G,CisLink-V2G,ENCODE-rE2G Extended,ENCODE-rE2G,ABC,Distance
0,K562,1895,115,0.688,0.578,0.678,0.632,0.519,0.366
1,HCT116,373,34,0.748,0.559,NaN,0.593,NaN,0.324
2,GM12878,68,16,0.912,0.852,0.669,0.653,0.731,0.853


In [2]:
ci = []
for c, p in P.items():
    for m in ("CisLink-E2G", "CisLink-V2G"):
        a, lo, hi = ev.auprc_ci(p.label, p[m], p.cluster)
        ci.append({"set": c, "model": m, "AUPRC": a, "lo": lo, "hi": hi})
pd.DataFrame(ci).round(3)

,set,model,AUPRC,lo,hi
0,K562,CisLink-E2G,0.688,0.592,0.773
1,K562,CisLink-V2G,0.578,0.467,0.690
2,HCT116,CisLink-E2G,0.748,0.646,0.897
3,HCT116,CisLink-V2G,0.559,0.342,0.754
4,GM12878,CisLink-E2G,0.912,0.778,1.000
5,GM12878,CisLink-V2G,0.852,0.687,1.000


## Paired differences on the same pairs

In [3]:
d = []
for c, p in P.items():
    for a, b in (("CisLink-E2G", "ENCODE-rE2G Extended"), ("CisLink-E2G", "ENCODE-rE2G"), ("CisLink-E2G", "CisLink-V2G"), ("CisLink-E2G", "Distance")):
        if p[b].isna().all():
            continue
        diff, lo, hi = ev.paired_difference(p.label, p[a], p[b].fillna(0), p.cluster)
        d.append({"set": c, "comparison": f"{a} - {b}", "difference": diff, "lo": lo, "hi": hi})
pd.DataFrame(d).round(3)

,set,comparison,difference,lo,hi
0,K562,CisLink-E2G - ENCODE-rE2G Extended,0.010,-0.055,0.080
1,K562,CisLink-E2G - ENCODE-rE2G,0.056,-0.010,0.126
2,K562,CisLink-E2G - CisLink-V2G,0.110,0.046,0.164
3,K562,CisLink-E2G - Distance,0.322,0.223,0.402
4,HCT116,CisLink-E2G - ENCODE-rE2G,0.155,0.055,0.302
5,HCT116,CisLink-E2G - CisLink-V2G,0.188,0.058,0.352
6,HCT116,CisLink-E2G - Distance,0.424,0.284,0.599
7,GM12878,CisLink-E2G - ENCODE-rE2G Extended,0.243,0.016,0.421
8,GM12878,CisLink-E2G - ENCODE-rE2G,0.259,0.000,0.411
9,GM12878,CisLink-E2G - CisLink-V2G,0.060,-0.044,0.211


## Table 1c: held-out K562 pairs by element–TSS distance

CisLink-E2G's margin over distance and over CisLink-V2G is largest 10–200 kb from the TSS.

In [4]:
k = P["K562"].copy(); k["bin"] = ev.distance_bins(k.distance_to_TSS)
t = []
for lo_, hi_, b in ev.DISTANCE_BINS:
    q = k[k.bin == b]
    r = {"distance": b, "pairs": len(q), "positives": int(q.label.sum())}
    for m in ("CisLink-E2G", "CisLink-V2G", "ENCODE-rE2G Extended", "Distance"):
        r[m] = ev.auprc(q.label, q[m])
    for a, b2 in (("CisLink-E2G", "Distance"), ("CisLink-E2G", "CisLink-V2G")):
        diff, lo2, hi2 = ev.paired_difference(q.label, q[a], q[b2], q.cluster)
        r[f"{a.split('-')[1]} - {b2.replace('CisLink-', '')}"] = f"{diff:+.3f} [{lo2:+.3f}, {hi2:+.3f}]"
    t.append(r)
pd.DataFrame(t).round(3)

,distance,pairs,positives,CisLink-E2G,CisLink-V2G,ENCODE-rE2G Extended,Distance,E2G - Distance,E2G - V2G
0,<10 kb,31,17,0.919,0.769,0.913,0.753,"+0.167 [-0.023, +0.306]","+0.150 [-0.023, +0.340]"
1,10-50 kb,135,40,0.821,0.683,0.831,0.414,"+0.407 [+0.235, +0.524]","+0.137 [+0.035, +0.228]"
2,50-200 kb,396,36,0.683,0.588,0.659,0.158,"+0.525 [+0.347, +0.658]","+0.095 [+0.005, +0.166]"
3,>200 kb,1333,22,0.191,0.153,0.262,0.035,"+0.156 [+0.019, +0.358]","+0.039 [-0.033, +0.123]"


## Pooled over cell types not used for training

The manuscript pools four sets (HCT116, GM12878 and the two hFOB screens) as the positive-weighted mean of per-set AUPRC differences. With the two redistributable sets:

In [5]:
S = {c: P[c].rename(columns={"label": "y"}) for c in ("HCT116", "GM12878")}
for a, b in (("CisLink-E2G", "CisLink-V2G"), ("CisLink-E2G", "ENCODE-rE2G")):
    r = ev.pooled_difference(S, a, b)
    print(f"{a} - {b}: {r['difference']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}]  per set {', '.join(f'{k} {v:+.3f}' for k, v in r['per_set'].items())}")

CisLink-E2G - CisLink-V2G: +0.147 [+0.049, +0.267]  per set HCT116 +0.188, GM12878 +0.060


CisLink-E2G - ENCODE-rE2G: +0.188 [+0.071, +0.291]  per set HCT116 +0.155, GM12878 +0.259
